# Summary analysis for shoaling QTL mapping


- Adapted from ShoalingSelectionAnalysis_2025_slim_clean.ipynb
- For this to work, you need to adapt the folder where raw data and metaData live.
- Also, you need to clone the jlsocialbehavior repository and create the environment jlsocial from the yml file.

## Define input and output folders. Give metadata location

In [ ]:
# Find the username dynamically, this will allow the script to work in different computers without hardcoding the username
import os

import getpass
username = getpass.getuser()
print(username)


In [ ]:

#define input and ouitput folders. Give location of metadata and the file name

# base = 'Y://Johannes//b//2019//' This is the original base folder where the meta data was stored
metaFolder = '//nasdcsr.unil.ch/RECHERCHE/FAC/FBM/CIG/jlarsch/default/D2c/07_Data/Carlos/Behavior/QTL/'  # Updated base folder. Metadata should live here.
codeDir = 'C:/Users/crodri19/larschpostdoc_analyses/selection_shoaling/jlsocialbehavior/' #adapt this to your code folder
metaFile='MetaData_QTL.xlsx'
ProcessingDir = '//nasdcsr.unil.ch/RECHERCHE/FAC/FBM/CIG/jlarsch/default/D2c/03_Common_Use//temp/shoaling_selection/ShoalQTL_temp_processing/'
outputDir = '//nasdcsr.unil.ch/RECHERCHE/FAC/FBM/CIG/jlarsch/default/D2c/03_Common_Use//temp/shoaling_selection/ShoalQTL_temp_output/'

# Inclusive lower date bound for QTL data used in plots. Set to None to include all dates.
QTL_INCLUDE_FROM_DATE = '2026-06-01'

print('searching for meta info here: ' + metaFolder)

os.chdir(codeDir)

## Import necessary python libraries

In [ ]:

# Import necessary python libraries


import matplotlib.pyplot as plt
import numpy as np
import statsmodels.stats.api as sms

%config InteractiveShellApp.pylab_import_all = False
%matplotlib inline
%reload_ext autoreload
%autoreload 2

import sys
import fnmatch

import math
import warnings
import matplotlib.pyplot as plt
import matplotlib as mpl
import pandas as pd
from pandas import DataFrame, Series
import seaborn as sns
import glob
#import h5py
from datetime import datetime
import PythonMeta as PMA
from matplotlib.ticker import AutoLocator

import functions.matrixUtilities_joh as mu
import functions.notebookHelper as nh
import functions.metaTree as mt

import models.experiment as xp
import models.experiment_set as es
import functions.paperFigureProps as pfp

from scipy import stats

pfp.paper()

## Load experiment and animal metadata


In [ ]:
# Check if the meta folder exists and list files
files = os.listdir('//nasdcsr.unil.ch/RECHERCHE/FAC/FBM/CIG/jlarsch/default/D2c/07_Data/Carlos/Behavior/QTL')
print(files)


# Save sheet with experiment info in table 'info'
info=pd.read_excel(metaFolder+metaFile, sheet_name='AllExp')
info

In [ ]:
# infoAn=pd.read_excel(metaFolder+metaFile, sheet_name='AllAn',parse_dates=['bd','expDate'])
# infoAn['bd'] = pd.to_datetime(infoAn['bd'], format='%Y%m%d')
# infoAn['expDate'] = pd.to_datetime(infoAn['expDate'], format='%Y%m%d')
# infoAn.tail()

#Save metadata sheet with individual animal info into the 'infoAn' table

infoAn = pd.read_excel(metaFolder + metaFile, sheet_name='AllAn')
infoAn['bd'] = pd.to_datetime(infoAn['bd'], format='%d-%m-%Y', dayfirst=True)
infoAn['expDate'] = pd.to_datetime(infoAn['expDate'], format='%d-%m-%Y', dayfirst=True)
# Exclude the new NA genotype everywhere by removing it before any downstream merge or plot.
infoAn = infoAn.loc[infoAn['genotype'].notna() & infoAn['genotype'].astype(str).str.lower().ne('na')].copy()
infoAn.tail(150)

### Data Points per Line


In [ ]:
# Count every retained AllAn metadata row with an assigned selection line.
required_line_count_columns = {'line', 'genotype'}
missing_line_count_columns = required_line_count_columns - set(infoAn.columns)
if missing_line_count_columns:
    raise ValueError('Animal metadata is missing required columns: ' + ', '.join(sorted(missing_line_count_columns)))

genotype_labels = infoAn['genotype'].astype('string').str.strip()
line_labels = infoAn['line'].astype('string').str.strip()
eligible_line_rows = (
    genotype_labels.notna()
    & genotype_labels.str.lower().ne('na').fillna(False)
    & line_labels.notna()
    & line_labels.ne('').fillna(False)
)

data_points_per_line = (
    pd.DataFrame({'line': line_labels[eligible_line_rows]})
    .groupby('line', sort=True)
    .size()
    .rename('n_fish')
    .reset_index()
)

data_points_per_line = pd.concat(
    [
        data_points_per_line,
        pd.DataFrame({
            'line': ['Total'],
            'n_fish': [data_points_per_line['n_fish'].sum()],
        }),
    ],
    ignore_index=True,
)
display(data_points_per_line)


In [ ]:
infoAn.genotype.unique()

## Prepare processing settings


In [ ]:
# collect video (?) meta information and save to new csv file for batch processing

aviPath=[]
posPath=[]
PLPath=[]
expTime = []
birthDayAll=[]
anIDsAll=[]
camHeightAll=[]

camHeight=[105,180] # these are outdated values, but we keep them for compatibility with old data

for index,row in info.iterrows():

    startDir=row.path+'\\'+row.folder+'\\'
    print('processing: ' + startDir)
    if not os.path.exists(startDir):
        print('WARNING: path does not exist: ' + startDir)
        continue

    
    posPath.append(glob.glob(startDir+'PositionTxt*')[0]) #this is the trajectory file
    PLPath.append(glob.glob(startDir+'PL*')[0]) # this is the pair list file
    
    head, tail = os.path.split(posPath[-1])
    currTime='dummy' #datetime.strptime(tail[-23:-4], '%Y-%m-%dT%H_%M_%S')
    expTime.append(currTime)
    
    camHeightAll.append(camHeight[('_dn_' in head)*1]) ######### this needs to be adapted for new data, since the camHeight is not stored in the meta data anymore.
    
    anNrs=row.anNr #Note that anNrs are 1 based!
    if ':' in anNrs:
        a,b=anNrs.split(sep=':')
        anNrs=np.arange(int(a),int(b)+1)
    else:
        anNrs=np.array(anNrs.split()).astype(int)
        
    anIDs=anNrs #-1 no more 0-based since using pandas merge to find animal numbers
    anIDsAll.extend(anIDs)

    bd=infoAn[infoAn.anNr.isin(anIDs)].bd.values.astype(str) # get birth dates of animals in this experiment
    #bd=infoAn.bd.values[anIDs-1] #a bit dirty to use anIDs directly here. Should merge
    birthDayAll.append(' '.join(list(bd)))

info['camHeight']=camHeightAll
info['txtPath']=posPath
info['pairList']=PLPath
info['aviPath']='default'
info['birthDayAll']=birthDayAll
info['epiDur'] = 5      # duration of individual episodes (default: 5 minutes)
info['episodes'] = 24   # number of episodes to process: -1 to load all episodes (default: -1)
info['inDish'] = 10#np.arange(len(posPath))*120     # time in dish before experiments started (default: 10)
info['arenaDiameter_mm'] = 70 # arena diameter (default: 100 mm)
info['minShift'] = 60 # minimum number of seconds to shift for control IAD
info['episodePLcode'] = 0 # flag if first two characters of episode name encode animal pair matrix (default: 0)
info['recomputeAnimalSize'] = 1 # flag to compute animals size from avi file (takes time, default: 1)
info['SaveNeighborhoodMaps'] = 0 # flag to save neighborhood maps for subsequent analysis (takes time, default: 1)
info['computeLeadership'] = 0 # flag to compute leadership index (takes time, default: 1)
info['ComputeBouts'] = 1 # flag to compute swim bout frequency (takes time, default: 1)
#info['set'] = np.arange(len(posPath))   # experiment set: can label groups of experiments (default: 0)
info['ProcessingDir']=ProcessingDir
info['outputDir']=outputDir
info['expTime']=expTime
info['readLim'] = 24*5*60*30+11

In [ ]:
csvFile=os.path.join(ProcessingDir,'processingSettings.csv')
info.to_csv(csvFile,encoding='utf-8')
info.tail()

## Read experiment data and processed shoaling summaries


In [ ]:
rereadData=1
if rereadData:
    def readExperiment(keepData=True):
        tmp=es.experiment_set(csvFile=csvFile,MissingOnly=True)
        if keepData:
            return tmp
        else:
            return 1

    expSet=readExperiment(keepData=False)

In [ ]:
csvPath = []
for f in [mu.splitall(x)[-1][:-4] for x in info.txtPath]:
    csvPath.append(glob.glob(ProcessingDir+f+'*siSummary*.csv')[0])

df=pd.DataFrame()
i=0
for fn in csvPath:
    print(fn)
    tmp=pd.read_csv(fn,index_col=0,sep=',')
    tmp.animalSet=i
    tmp.animalIndex=tmp.animalIndex+((i)*35)
    tmp.animalIndex=np.array(anIDsAll)[tmp.animalIndex]
    df=pd.concat([df,tmp])
    i+=1
df['episode']=[x.strip().replace('_','') for x in df['episode']]
df=pd.merge(df,infoAn[['anNr','line','genotype']],left_on='animalIndex',right_on='anNr',how='left')
df=pd.merge(df,info[['date']],left_on='animalSet',right_on=info.index,how='left')
df['setup'] = info['setup'].values[df['animalSet'].values]
# Drop rows that belong to the NA genotype before building downstream experiment labels.
df = df.loc[
    df['genotype'].notna()
    & df['genotype'].astype(str).str.lower().ne('na')
    & df['line'].notna()
    & df['date'].notna()
].copy()

# Parse dates separately so filtering does not change the existing date or lineSet schemas.
qtl_experiment_dates = pd.to_datetime(df['date'], dayfirst=True, errors='coerce')
invalid_qtl_dates = qtl_experiment_dates.isna()
if invalid_qtl_dates.any():
    invalid_date_values = df.loc[invalid_qtl_dates, 'date'].astype(str).unique()
    invalid_date_preview = ', '.join(invalid_date_values[:5])
    raise ValueError(
        f'Cannot apply the QTL plot date filter because experiment dates could not be parsed: {invalid_date_preview}'
    )

if QTL_INCLUDE_FROM_DATE is None:
    qtl_include_from_date = None
else:
    try:
        qtl_include_from_date = pd.to_datetime(
            QTL_INCLUDE_FROM_DATE, format='%Y-%m-%d', errors='raise'
        )
    except (TypeError, ValueError) as exc:
        raise ValueError(
            'QTL_INCLUDE_FROM_DATE must be None or a valid YYYY-MM-DD date.'
        ) from exc

    df = df.loc[qtl_experiment_dates >= qtl_include_from_date].copy()
    if df.empty:
        raise ValueError(
            f'No QTL data remain on or after {qtl_include_from_date.date()}.'
        )

filter_label = (
    'all dates'
    if qtl_include_from_date is None
    else f'on or after {qtl_include_from_date.date()}'
)
print(
    f'QTL plot date filter: {filter_label}; retained '
    f'{df["animalSet"].nunique()} experiments and {len(df)} rows.'
)
print('df shape',df.shape)



# Filtered data counts per line

In [ ]:
# Count each fish once using only the QTL rows retained by the date filter.
qtl_data_points_per_line = (
    df[['animalIndex', 'line']]
    .drop_duplicates()
    .groupby('line', sort=True)
    .size()
    .rename('n_fish')
    .reset_index()
)

qtl_data_points_per_line = pd.concat(
    [
        qtl_data_points_per_line,
        pd.DataFrame({
            'line': ['Total'],
            'n_fish': [qtl_data_points_per_line['n_fish'].sum()],
        }),
    ],
    ignore_index=True,
)
display(qtl_data_points_per_line)

df['lineSet']=[x+'_'+y for x,y in zip(df.line, df.date)]


In [ ]:
#df.tail(40)
df
#df.animalSet.unique()

## Build episode-level animal summaries


In [ ]:
sns.set_palette('viridis',3)
co=sns.color_palette("viridis", 3)
idx=(df['inDishTime']<240) & (df['inDishTime']>80) # PUT THIS SETTING ON THE TOP - ANALYSIS WINDOW BETWEEN 80 AND 240s
dfDR=df[idx]
dfEpiAn=dfDR.groupby(['episode','animalIndex','line','setup','genotype','date','lineSet'],sort=True).mean(numeric_only=True).reset_index()

In [ ]:
#dfEpiAn.head()
dfEpiAn

## Summary statistic helpers


In [ ]:

def sem(x):
    return np.std(x, ddof=1) / np.sqrt(len(x))

x=np.random.random(100)

def ci95(x):
    return np.nanmean(x)-sms.DescrStatsW(x[np.isfinite(x)]).tconfint_mean()[0]

print('std of uniform = 0.2886751345948129. STDdata:',np.std(x), 'semData:',sem(x),'samples:',x.shape) 
print('ci95:',ci95(x))

In [ ]:
dfPlot=(df.groupby(['inDishTime','episode','genotype','lineSet']).si.agg(['mean','std',sem,ci95])
    .unstack()
    .stack(dropna=True)
    .reset_index())

dfPlot.head()

## Shoaling index analyses

### Shoaling index throughout the experiment - linear vs bout

In [ ]:
dfPlot_filtered = dfPlot[dfPlot['genotype'] == 'F2']
#dfPlot_filtered = dfPlot[~dfPlot['genotype'].str.contains(r'^(esc_(lo|hi)|na)$', regex=True)]


markers = {'F2': 'o'}  # "o" is a fake genotype, i get an error if I only use one marker


g = sns.FacetGrid(
    dfPlot_filtered,
    col='lineSet',         # Facet by genotype (change as needed)
    col_wrap=3,             # Adjust for compactness
    sharex=True,
    sharey=True,
    height=2,
    aspect=1
)
g.map_dataframe(
    sns.scatterplot,
    x='inDishTime',
    y='mean',
    hue='episode',
    style='genotype',
    size=10,
    markers=markers,
    legend='full',

)
g.set(xlim=(0, 2.5*60), ylim=(0, .5))
g.set_axis_labels('Time (Minutes)', 'Attraction')
g.set_titles('{col_name}')
g.figure.subplots_adjust(top=0.85)
g.figure.suptitle('Mean attraction, all animals', fontsize=14)

# Remove the default legend and add a correct one
for ax in g.axes.flatten():
    handles, labels = ax.get_legend_handles_labels()
    ax.legend_.remove() if hasattr(ax, "legend_") and ax.legend_ else None

# Add a single legend to the figure
handles, labels = g.axes[0].get_legend_handles_labels()
g.figure.legend(handles, labels, ncol=1, handletextpad=0, bbox_to_anchor=(1, 1), loc='upper right')

# Save the plot using the path variable
g.figure.savefig(os.path.join(outputDir, "attraction_linearvsbout.pdf"), bbox_inches="tight")

outputDir

### Shoaling index between setups

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

sns.pointplot(data=dfEpiAn[dfEpiAn.episode=='02k20f'],x='lineSet',y='si',hue='setup', dodge=0.4, linestyle='none')
plt.ylim([0,.4])
plt.xticks(rotation=90);
plt.ylabel('Shoaling Index');

# Save the plot using the path variable
plt.savefig(os.path.join(outputDir, "shoalindex_setups.pdf"), bbox_inches="tight")



### Shoaling index across experiments

In [ ]:
fig, axes = plt.subplots(figsize=(8,3))

#All data
#ix=(dfEpiAn.episode=='02k20f')

#exclude esc and 01k01f
ix=(dfEpiAn.episode=='02k20f')&((dfEpiAn.genotype=='F2')) 


selDat=dfEpiAn[ix]

#Settings

show_lines = "All" 


if show_lines != "All":
    selDat = selDat[selDat['line'].isin(show_lines)]

# Define custom colors for each genotype
genotype_hue_order = ['F2']
custom_palette = {'F2': "#1F77B4"}

sns.pointplot(data=selDat,
              x='lineSet',
              y='si',
              hue='genotype',
             linestyle='-',
             errorbar='sd',
             hue_order=genotype_hue_order,
             palette=custom_palette)

sns.swarmplot(data=selDat,
              x='lineSet',
              y='si',
              hue='genotype',
              zorder=-1,
              size=5,
              alpha=.5,
              #order=allCat,
             #hue_order=["hi", "lo",'wt','esc','mid'])
             hue_order=genotype_hue_order,
             palette=custom_palette)

sns.despine()

axes.set_ylabel('Shoaling Index')
axes.axhline(0,ls=':',color='k')
axes.set_title('Shoaling Index across experiments');
plt.legend(title='line',ncol=2,handletextpad=0,bbox_to_anchor=(1, 1.05))
plt.xticks(rotation=90);




In [ ]:
dfEpiAn.line.unique() #check existing lines

### SI between normal and escapee fish

In [ ]:
fig, axes = plt.subplots(figsize=(8, 3))
# Set this to True when you want to plot only the F2 samples and only show the F2 group size.
plot_only_F2 = False
plot_genotypes = ['F2'] if plot_only_F2 else ['F2', 'esc']
pointplot_dodge = 0 if plot_only_F2 else .4
ix=(dfEpiAn.episode=='02k20f')&(dfEpiAn.genotype.isin(plot_genotypes))

selDat=dfEpiAn[ix]

#Settings

show_lines = "All" 
xlegend_orientation = 'vertical' #choose verticar or horizontal


if show_lines != "All":
    selDat = selDat[selDat['line'].isin(show_lines)]

allCat=selDat.lineSet.unique()
allCat.sort()
allCat=allCat[::-1]

# Count the number of unique animals per experiment and genotype for the labels.
# These counts are used to annotate each cohort with the group size(s) above the plot.
group_columns = ['F2'] if plot_only_F2 else ['F2', 'esc']
group_sizes = (
    selDat.groupby(['lineSet', 'genotype'])['animalIndex']
    .nunique()
    .unstack(fill_value=0)
    .reindex(columns=group_columns, fill_value=0)
    .fillna(0)
    .astype(int)
)

sns.swarmplot(data=selDat,
              x='lineSet',
              y='si',
              hue='genotype',
              zorder=-1,
              dodge=.5,
              size=5,
              alpha=.5,
              #order=allCat,
             #hue_order=["hi", "lo",'wt','esc','mid'])
             hue_order=plot_genotypes)

sns.pointplot(data=selDat,
              x='lineSet',
              y='si',
              hue='genotype',
              dodge=pointplot_dodge,
              # Draw only the summary points and error bars, without connecting lines across experiments.
              linestyle='none',
              errorbar='sd',
              hue_order=plot_genotypes)



sns.despine()
axes.set_ylim([-0.1,0.6])

axes.set_ylabel('Attraction')
axes.set_xlabel('Cohort')

axes.axhline(0,ls=':',color='k')
#axes.set_title('Selection F1');

#Write the x-axis labels with the cohort names, and rotate them if needed to keep them readable.
# Read the current category labels after the seaborn plot has created the x-axis ticks.
xticklabels = [t.get_text() for t in axes.get_xticklabels()]
xticks = axes.get_xticks()
axes.set_xticks(xticks)
if xlegend_orientation == 'horizontal':
    # Keep labels flat when the layout is horizontal.
    axes.set_xticklabels(xticklabels, rotation=0)
else:
    # Rotate labels to keep long cohort names readable.
    axes.set_xticklabels(xticklabels, rotation=90)

# Annotate each cohort with the number of animals in each genotype group above the plot.
for xpos, label in zip(xticks, xticklabels):
    if label in group_sizes.index:
        f2_n = group_sizes.loc[label, 'F2'] if 'F2' in group_sizes.columns else 0
        if plot_only_F2:
            label_text = f'F2={f2_n}'
        else:
            esc_n = group_sizes.loc[label, 'esc'] if 'esc' in group_sizes.columns else 0
            label_text = f'F2={f2_n}\nesc={esc_n}'
        axes.text(
            xpos,
            1.00,
            label_text,
            transform=axes.get_xaxis_transform(),
            ha='center',
            va='bottom',
            fontsize=9,
        )

# Keep the title above the group-size annotations, with extra padding so they do not crowd each other.
plt.title('Selection per experiment', pad=36);


handles, labels = axes.get_legend_handles_labels()
unique_handles = []
unique_labels = []
for handle, label in zip(handles, labels):
    if label not in unique_labels:
        unique_handles.append(handle)
        unique_labels.append(label)

l = plt.legend(unique_handles, unique_labels, title='Genotype', ncol=1, handletextpad=0,
               bbox_to_anchor=(1.15, 1.05),
              frameon=False)

#figPath=base+'SelectionAllToF1.png'
#plt.savefig(figPath,bbox_inches='tight')


# Save the plot using the path variable

if show_lines == "All":
    plt.savefig(os.path.join(outputDir, "shoalindex_hivslo.pdf"), bbox_inches="tight")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.5), sharey=True)
# Set this to True when you want to plot only the F2 samples and only show the F2 group size.
plot_only_F2 = True
plot_genotypes = ['F2'] if plot_only_F2 else ['F2', 'esc']
ix=(dfEpiAn.episode=='02k20f')&(dfEpiAn.genotype.isin(plot_genotypes))

selDat=dfEpiAn[ix]

#Parse date as datetime for sorting and ordering of cohorts in the plot
selDat = selDat.copy()
selDat['_plot_date'] = pd.to_datetime(
    selDat['date'],
    format='%d-%m-%Y',
    errors='raise',
)

line_palette = {
    'Sel1A': "#E29844",
    'Sel1B': "#E2551E",
    'Sel3A': "#163F97",
    'Sel3C': "#7555A9",
    'Sel3D': "#2C7938",
}

family_palette = {
    'Sel1': "#CE9223",
    'Sel3': '#1B5699',
}

# Settings
show_lines = "All" 
xlegend_orientation = 'vertical' #choose verticar or horizontal


if show_lines != "All":
    selDat = selDat[selDat['line'].isin(show_lines)].copy()

selDat['selection_family'] = selDat['line'].str.extract(r'^(Sel[13])', expand=False)

# Count the number of unique animals per experiment and genotype for the labels.
# These counts are used to annotate each cohort with the group size(s) above the plot.
group_columns = ['F2'] if plot_only_F2 else ['F2', 'esc']
group_sizes = (
    selDat.groupby(['lineSet', 'genotype'])['animalIndex']
    .nunique()
    .unstack(fill_value=0)
    .reindex(columns=group_columns, fill_value=0)
    .fillna(0)
    .astype(int)
)

families = ['Sel1', 'Sel3']
family_means = {}
all_handles = []
all_labels = []

for ax, family in zip(axes, families):
    family_data = selDat[selDat['selection_family'] == family].copy()
    family_order = (
        family_data[['line', '_plot_date', 'lineSet']]
        .drop_duplicates()
        .sort_values(['line', '_plot_date', 'lineSet'])['lineSet']
        .tolist()
    )

    sns.swarmplot(
        data=family_data,
        x='lineSet',
        y='si',
        hue='line',
        palette=line_palette,
        order=family_order,
        dodge=False,
        zorder=-1,
        size=5,
        alpha=.5,
        ax=ax,
    )

    sns.pointplot(
        data=family_data,
        x='lineSet',
        y='si',
        hue='line',
        palette=line_palette,
        order=family_order,
        dodge=False,
        linestyle='none',
        errorbar='sd',
        ax=ax,
    )

    # Give each cohort equal weight in the family-level average.
    family_mean = family_data.groupby('lineSet')['si'].mean().mean()
    family_means[family] = family_mean
    ax.hlines(
        family_mean,
        xmin=-0.4,
        xmax=len(family_order) - 0.6,
        color=family_palette[family],
        linewidth=2.5,
        linestyle='--',
        zorder=4,
    )

    ax.set_ylim([-0.1, 0.6])
    ax.set_xlabel('Cohort')
    ax.axhline(0, linestyle=':', color='k')
    ax.set_title(family, pad=36, fontweight='bold', fontsize=20)

    xticklabels = [tick.get_text() for tick in ax.get_xticklabels()]
    xticks = ax.get_xticks()
    ax.set_xticks(xticks)
    rotation = 0 if xlegend_orientation == 'horizontal' else 90
    ax.set_xticklabels(xticklabels, rotation=rotation)

    # Annotate each cohort with its number of F2 animals.
    for xpos, label in zip(xticks, xticklabels):
        if label in group_sizes.index:
            f2_n = group_sizes.loc[label, 'F2'] if 'F2' in group_sizes.columns else 0
            ax.text(
                xpos,
                1.00,
                f'F2={f2_n}',
                transform=ax.get_xaxis_transform(),
                ha='center',
                va='bottom',
                fontsize=9,
            )

    handles, labels = ax.get_legend_handles_labels()
    all_handles.extend(handles)
    all_labels.extend(labels)
    if ax.legend_ is not None:
        ax.legend_.remove()

axes[0].set_ylabel('Shoaling Index')
axes[1].set_ylabel('')
sns.despine(fig=fig)
fig.subplots_adjust(wspace=0.08)

unique_handles = []
unique_labels = []
for handle, label in zip(all_handles, all_labels):
    if label not in unique_labels:
        unique_handles.append(handle)
        unique_labels.append(label)

fig.legend(
    unique_handles,
    unique_labels,
    title='Line',
    loc='upper left',
    bbox_to_anchor=(0.90, 0.92),
    frameon=False,
)

fig.text(
    0.91, 0.43,
    f"Sel1 mean = {family_means['Sel1']:.3f}",
    color=family_palette['Sel1'],
    fontweight='bold',
)
fig.text(
    0.91, 0.35,
    f"Sel3 mean = {family_means['Sel3']:.3f}",
    color=family_palette['Sel3'],
    fontweight='bold',
)

if show_lines == "All":
    plt.savefig(os.path.join(outputDir, "shoalindex_hivslo.pdf"), bbox_inches="tight")


### Checking delta SI between bout and linear dots

In [ ]:
# ix=((dfEpiAn.genotype=='F2')) 

# mainDat=dfEpiAn[ix]


#Extract delta of shoaling index between linear and bout experiments
linear_data=dfEpiAn[dfEpiAn.episode=='01k01f']
bout_data=dfEpiAn[dfEpiAn.episode=='02k20f']

delta_data = bout_data.copy()
delta_data['si'] = bout_data['si'].values - linear_data['si'].values

delta_data['si'].values


fig, axes = plt.subplots(figsize=(8, 3))

selDat=delta_data
show_lines = "All" 
#show_lines = [ "Sel1_10", "Sel1_11"]  # set the lines you want to keep

if show_lines != "All":
    selDat = selDat[selDat['line'].isin(show_lines)]

allCat=selDat.lineSet.unique()
allCat.sort()
allCat=allCat[::-1]

sns.swarmplot(data=selDat,
              x='lineSet',
              y='si',
              hue='genotype',
              zorder=-1,
              dodge=.5,
              size=5,
              alpha=.5,
              #order=allCat,
             #hue_order=["hi", "lo",'wt','esc','mid'])
             )

sns.pointplot(data=selDat,
              x='lineSet',
              y='si',
              hue='genotype',
              #dodge=.4,
              linestyle='none',
              errorbar='sd',
              #hue_order=["hi", "lo",'wt','esc','mid'])
              )


sns.despine()

axes.set_ylabel('Delta SI (bout-linear)')
axes.set_xlabel('Cohort')

axes.axhline(0,ls=':',color='k')
#axes.set_title('Selection F1');

plt.xticks(rotation=90);

handles, labels = axes.get_legend_handles_labels()

l = plt.legend(handles[0:5], labels[0:5], title='Parents',ncol=1,handletextpad=0,
               bbox_to_anchor=(1, 1.05),
              frameon=False)

#figPath=base+'SelectionAllToF1.png'
#plt.savefig(figPath,bbox_inches='tight')

plt.title('Delta SI per experiment');

# Save the plot using the path variable
plt.savefig(os.path.join(outputDir, "shoalindelta_hivslo.pdf"), bbox_inches="tight")


### Delta SI by selection family

In [ ]:
# Plot the existing bout-minus-linear Delta SI metric for F2 cohorts by selection family.
delta_plot_data = delta_data[delta_data['genotype'] == 'F2'].copy()
delta_plot_data['_plot_date'] = pd.to_datetime(
    delta_plot_data['date'],
    format='%d-%m-%Y',
    errors='raise',
)
delta_plot_data['selection_family'] = delta_plot_data['line'].str.extract(
    r'^(Sel[13])',
    expand=False,
)

# delta_line_palette = {
#     'Sel1A': '#3B6FB6',
#     'Sel1B': '#79A7D3',
#     'Sel3A': '#238B6F',
#     'Sel3C': '#55A98B',
#     'Sel3D': '#8BC9AE',
# }
# delta_family_palette = {
#     'Sel1': '#244E78',
#     'Sel3': '#176B57',
# }

delta_line_palette = {
    'Sel1A': "#E29844",
    'Sel1B': "#E2551E",
    'Sel3A': "#163F97",
    'Sel3C': "#7555A9",
    'Sel3D': "#2C7938",
}

delta_family_palette = {
    'Sel1': "#CE9223",
    'Sel3': "#1B5699",
}


delta_families = ['Sel1', 'Sel3']
delta_show_lines = 'All'
delta_xlegend_orientation = 'vertical'

if delta_show_lines != 'All':
    delta_plot_data = delta_plot_data[
        delta_plot_data['line'].isin(delta_show_lines)
    ].copy()

missing_delta_lines = (
    set(delta_plot_data['line'].dropna().unique()) - set(delta_line_palette)
)
if missing_delta_lines:
    raise ValueError(f'Missing Delta SI colors for lines: {sorted(missing_delta_lines)}')

delta_group_sizes = (
    delta_plot_data.groupby('lineSet')['animalIndex']
    .nunique()
    .astype(int)
)

fig, axes = plt.subplots(1, 2, figsize=(13, 3.5), sharey=True)
delta_family_means = {}
delta_legend_handles = []
delta_legend_labels = []

for ax, family in zip(axes, delta_families):
    family_data = delta_plot_data[
        delta_plot_data['selection_family'] == family
    ].copy()
    family_order = (
        family_data[['line', '_plot_date', 'lineSet']]
        .drop_duplicates()
        .sort_values(['line', '_plot_date', 'lineSet'])['lineSet']
        .tolist()
    )

    sns.swarmplot(
        data=family_data,
        x='lineSet',
        y='si',
        hue='line',
        palette=delta_line_palette,
        order=family_order,
        dodge=False,
        zorder=-1,
        size=5,
        alpha=.5,
        ax=ax,
    )

    sns.pointplot(
        data=family_data,
        x='lineSet',
        y='si',
        hue='line',
        palette=delta_line_palette,
        order=family_order,
        dodge=False,
        linestyle='none',
        errorbar='sd',
        ax=ax,
    )

    # Average cohort means so each cohort contributes equally to its family mean.
    family_mean = family_data.groupby('lineSet')['si'].mean().mean()
    delta_family_means[family] = family_mean
    ax.hlines(
        family_mean,
        xmin=-0.4,
        xmax=len(family_order) - 0.6,
        color=delta_family_palette[family],
        linewidth=2.5,
        linestyle='--',
        zorder=4,
    )

    ax.set_ylim([-0.15, 0.6])
    ax.set_xlabel('Cohort')
    ax.axhline(0, linestyle=':', color='k')
    ax.set_title(family, fontsize=16, fontweight='bold', pad=36)

    xticklabels = [tick.get_text() for tick in ax.get_xticklabels()]
    xticks = ax.get_xticks()
    ax.set_xticks(xticks)
    rotation = 0 if delta_xlegend_orientation == 'horizontal' else 90
    ax.set_xticklabels(xticklabels, rotation=rotation)

    for xpos, label in zip(xticks, xticklabels):
        if label in delta_group_sizes.index:
            ax.text(
                xpos,
                1.00,
                f'F2={delta_group_sizes.loc[label]}',
                transform=ax.get_xaxis_transform(),
                ha='center',
                va='bottom',
                fontsize=9,
            )

    handles, labels = ax.get_legend_handles_labels()
    delta_legend_handles.extend(handles)
    delta_legend_labels.extend(labels)
    if ax.legend_ is not None:
        ax.legend_.remove()

axes[0].set_ylabel('Delta SI (bout-linear)')
axes[1].set_ylabel('')
sns.despine(fig=fig)
fig.subplots_adjust(wspace=0.08)

unique_delta_handles = []
unique_delta_labels = []
for handle, label in zip(delta_legend_handles, delta_legend_labels):
    if label not in unique_delta_labels:
        unique_delta_handles.append(handle)
        unique_delta_labels.append(label)

fig.legend(
    unique_delta_handles,
    unique_delta_labels,
    title='Line',
    loc='upper left',
    bbox_to_anchor=(0.91, 0.92),
    frameon=False,
)
fig.text(
    0.91, 0.43,
    f"Sel1 mean = {delta_family_means['Sel1']:.3f}",
    color=delta_family_palette['Sel1'],
    fontweight='bold',
)
fig.text(
    0.91, 0.35,
    f"Sel3 mean = {delta_family_means['Sel3']:.3f}",
    color=delta_family_palette['Sel3'],
    fontweight='bold',
)

plt.savefig(
    os.path.join(outputDir, 'shoalindelta_sel1_sel3.pdf'),
    bbox_inches='tight',
)


## Checking size, speed and SI across experiments

### Fish size across experiments

In [ ]:
fig, axes = plt.subplots(figsize=(8, 3))
#ix=(dfEpiAn.episode=='02k20f')&((dfEpiAn.genotype=='hi') | (dfEpiAn.genotype=='lo'))
ix=(dfEpiAn.episode=='02k20f')

selDat=dfEpiAn[ix]

show_lines = "All" 
#show_lines = [ "Sel1_10", "Sel1_11"]  # set the lines you want to keep

if show_lines != "All":
    selDat = selDat[selDat['line'].isin(show_lines)]

allCat=selDat.lineSet.unique()
allCat.sort()
allCat=allCat[::-1]

# sns.swarmplot(data=selDat,
#               x='lineSet',
#               y='anSize',
#               hue='genotype',
#               zorder=-1,
#               dodge=.5,
#               size=4,
#               alpha=.5,
#               #order=allCat,
#              #hue_order=["hi", "lo",'wt','esc','mid'])
#              hue_order=["hi",'esc_hi', "lo",'esc_lo','wt','mid'])

sns.pointplot(data=selDat,
              x='lineSet',
              y='anSize',
              hue='genotype',
              dodge=.1,
              linestyle='-',
              errorbar='sd',
              #hue_order=["hi", "lo",'wt','esc','mid'])
              hue_order=["F2",'esc'])


sns.despine()

axes.set_ylabel('anSize (mm)')
axes.set_xlabel('Cohort')

axes.axhline(0,ls=':',color='k')
#axes.set_title('Selection F1');

plt.xticks(rotation=90);

handles, labels = axes.get_legend_handles_labels()

l = plt.legend(handles[0:5], labels[0:5], title='Parents',ncol=1,handletextpad=0,
               bbox_to_anchor=(1, 1.05),
              frameon=False)

#figPath=base+'SelectionAllToF1.png'
#plt.savefig(figPath,bbox_inches='tight')

plt.title('animal sizes across experiments');

# Save the plot using the path variable
plt.savefig(os.path.join(outputDir, "anSize_hivslo.pdf"), bbox_inches="tight")



In [ ]:

# #Subset data for only experiments happening from August 2025
# dfEpiAn.date = pd.to_datetime(dfEpiAn.date, format='%d-%m-%Y', dayfirst=True)
# dfEpiAn.date.dtype

# date_cutoff = pd.to_datetime('01-08-2025', format='%d-%m-%Y', dayfirst=True)

# dfEpiAn[dfEpiAn.date>=date_cutoff]
dfEpiAn

### Fish speed across experiments

In [ ]:
fig, axes = plt.subplots(figsize=(8, 3))
#ix=(dfEpiAn.episode=='02k20f')&((dfEpiAn.genotype=='hi') | (dfEpiAn.genotype=='lo'))
#ix=(dfEpiAn.episode=='02k20f')&((dfEpiAn.genotype=='hi') | (dfEpiAn.genotype=='lo'))&(dfEpiAn.avgSpeed<10) #filter extreme speed values
ix=(dfEpiAn.episode=='02k20f')&(dfEpiAn.genotype=='F2')

selDat=dfEpiAn[ix]

show_lines = "All" 
#show_lines = [ "Sel1_10", "Sel1_11"]  # set the lines you want to keep

if show_lines != "All":
    selDat = selDat[selDat['line'].isin(show_lines)]

allCat=selDat.lineSet.unique()
allCat.sort()
allCat=allCat[::-1]

sns.swarmplot(data=selDat,
              x='lineSet',
              y='avgSpeed',
              hue='genotype',
              zorder=-1,
              dodge=.5,
              size=4,
              alpha=.5,
              #order=allCat,
             #hue_order=["hi", "lo",'wt','esc','mid'])
             hue_order=["F2"])

sns.pointplot(data=selDat,
              x='lineSet',
              y='avgSpeed',
              hue='genotype',
              #dodge=.5,
              linestyle='-',
              errorbar='sd',
              #hue_order=["hi", "lo",'wt','esc','mid'])
              hue_order=["F2"])


sns.despine()

axes.set_ylabel('average Speed (mm/s?)')
axes.set_xlabel('Cohort')

axes.axhline(0,ls=':',color='k')
#axes.set_title('Selection F1');

axes.set_ylim(0, 17)

plt.xticks(rotation=90);

handles, labels = axes.get_legend_handles_labels()

l = plt.legend(handles[0:5], labels[0:5], title='Parents',ncol=1,handletextpad=0,
               bbox_to_anchor=(1.07, 1.05),
              frameon=False)

#figPath=base+'SelectionAllToF1.png'
#plt.savefig(figPath,bbox_inches='tight')

plt.title('average speeds across experiments');

# Save the plot using the path variable
plt.savefig(os.path.join(outputDir, "avgSpeed_hivslo.pdf"), bbox_inches="tight")


### Thimogtaxis across experiments

In [ ]:
fig, axes = plt.subplots(figsize=(8, 3))
#ix=(dfEpiAn.episode=='02k20f')&((dfEpiAn.genotype=='hi') | (dfEpiAn.genotype=='lo'))#&(~dfEpiAn.lineSet.str.contains('_2_'))
ix=(dfEpiAn.episode=='02k20f')&(dfEpiAn.genotype=='F2')

selDat=dfEpiAn[ix]

show_lines = "All" 
#show_lines = [ "Sel1_10", "Sel1_11"]  # set the lines you want to keep

if show_lines != "All":
    selDat = selDat[selDat['line'].isin(show_lines)]

allCat=selDat.lineSet.unique()
allCat.sort()
allCat=allCat[::-1]

sns.swarmplot(data=selDat,
              x='lineSet',
              y='thigmoIndex',
              hue='genotype',
              zorder=-1,
              dodge=.5,
              size=4,
              alpha=.5,
              #order=allCat,
             #hue_order=["hi", "lo",'wt','esc','mid'])
             hue_order=["F2"])

sns.pointplot(data=selDat,
              x='lineSet',
              y='thigmoIndex',
              hue='genotype',
              linestyle='-',
              errorbar='sd',
              hue_order=["F2", "esc"]
              #hue_order=["hi",'esc_hi', "lo",'esc_lo','wt','mid'])
)

sns.despine()

axes.set_ylabel('Thigmotaxis Distance')
axes.set_xlabel('Cohort')

axes.axhline(0,ls=':',color='k')
#axes.set_title('Selection F1');
axes.set_ylim(0, 30)

plt.xticks(rotation=90);

handles, labels = axes.get_legend_handles_labels()

l = plt.legend(handles[0:5], labels[0:5], title='Parents',ncol=1,handletextpad=0,
               bbox_to_anchor=(1, 1.05),
              frameon=False)

#figPath=base+'SelectionAllToF1.png'
#plt.savefig(figPath,bbox_inches='tight')

plt.title('Thigmotaxis Distance across experiments');

# Save the plot using the path variable
plt.savefig(os.path.join(outputDir, "thigmoIndex_hivslo.pdf"), bbox_inches="tight")

## Correlations of SI with other metrics per lineSet

In [ ]:
# # Correlations with SI, split into one row per lineSet.
# si_line_set_comparisons = [
#     ('si', 'avgSpeed', 'Shoaling Index', 'Average Speed'),
#     ('si', 'anSize', 'Shoaling Index', 'Animal Size'),
#     ('si', 'thigmoIndex', 'Shoaling Index', 'Thigmotaxis Distance')
# ]

# # Hard-coded threshold guides for the grid plot.
# max_speed = 10
# min_speed = 1
# max_thigmoIndex = 30
# min_thigmoIndex = 10


# si_xlim = (-0.1, 0.6)  # change to whatever range you want


# line_set_data = (
#     dfEpiAn[ix]
#     .groupby(['animalIndex', 'genotype', 'lineSet'])
#     .mean(numeric_only=True)
#     .reset_index()
# )

# line_set_data = line_set_data.dropna(subset=['anSize', 'thigmoIndex', 'avgSpeed', 'si'])
# line_sets = sorted(line_set_data['lineSet'].dropna().unique())
# si_line_set_genotypes = [
#     genotype for genotype in genotypes
#     if genotype in line_set_data['genotype'].dropna().unique()
# ]

# nrows = len(line_sets)
# ncols = len(si_line_set_comparisons)
# fig, axes = plt.subplots(
#     nrows,
#     ncols,
#     figsize=(5 * ncols, 3.2 * nrows),
#     sharex='col',
#     sharey='col',
#     squeeze=False
# )
# fig.suptitle('Correlations of metrics with SI per lineSet', fontsize=14, y=0.995)

# for row_idx, line_set in enumerate(line_sets):
#     for col_idx, (x_var, y_var, x_label, y_label) in enumerate(si_line_set_comparisons):
#         ax = axes[row_idx, col_idx]
#         for genotype_idx, genotype in enumerate(si_line_set_genotypes):
#             plot_data = line_set_data[
#                 (line_set_data['lineSet'] == line_set)
#                 & (line_set_data['genotype'] == genotype)
#             ].dropna(subset=[x_var, y_var])
#             color = custom_palette.get(genotype, 'C0')

#             ax.scatter(plot_data[x_var], plot_data[y_var], s=35, alpha=0.7, color=color)

#             if (
#                 len(plot_data) > 1
#                 and plot_data[x_var].nunique() > 1
#                 and plot_data[y_var].nunique() > 1
#             ):
#                 slope, intercept, r_value, p_value, std_err = stats.linregress(
#                     plot_data[x_var], plot_data[y_var]
#                 )
#                 r_squared = r_value**2
#             else:
#                 r_squared = np.nan

#             r_squared_label = f'R^2 = {r_squared:.3f}' if np.isfinite(r_squared) else 'R^2 = N/A'
#             if len(si_line_set_genotypes) > 1:
#                 r_squared_label = f'{genotype}: {r_squared_label}'

#             ax.text(
#                 0.98,
#                 0.92 - 0.12 * genotype_idx,
#                 r_squared_label,
#                 transform=ax.transAxes,
#                 verticalalignment='top',
#                 horizontalalignment='right',
#                 color=color,
#                 bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5)
#             )

#         if y_var == 'avgSpeed':
#             ax.axhline(min_speed, color='red', linestyle='dotted', linewidth=1.5)
#             ax.axhline(max_speed, color='red', linestyle='dotted', linewidth=1.5)

#         if y_var == 'thigmoIndex':
#             ax.axhline(min_thigmoIndex, color='red', linestyle='dotted', linewidth=1.5)
#             ax.axhline(max_thigmoIndex, color='red', linestyle='dotted', linewidth=1.5)

#         if row_idx == 0:
#             ax.set_title(y_label, fontsize=12)
#         if row_idx == nrows - 1:
#             ax.set_xlabel(x_label)
#         else:
#             ax.set_xlabel('')

#         ax.set_ylabel(y_label)
#         ax.set_xlim(si_xlim)
#         if col_idx == 0:
#             ax.text(
#                 -0.35,
#                 0.5,
#                 line_set,
#                 transform=ax.transAxes,
#                 rotation=90,
#                 va='center',
#                 ha='center',
#                 fontsize=11,
#                 fontweight='bold'
#             )

# plt.tight_layout()
# plt.show()


# F2s vs selected lines metrics comparisons

## Importing data from selection lines experiments

In [ ]:
#Import data from selection experiments for comparison
selDir = '//nasdcsr.unil.ch/RECHERCHE/FAC/FBM/CIG/jlarsch/default/D2c/03_Common_Use//temp/shoaling_selection/ShoalSelec_temp_output/'
SelEpiAn = pd.read_csv(os.path.join(selDir, 'dfEpiAn_selection.csv'))

# Create a new column to indicate the dataset (F2 vs selection)
temp_dfEpiAn = dfEpiAn.copy()
temp_dfEpiAn['dataset'] = "F2"
SelEpiAn['dataset'] = "Selection"


# Combine the selection data with the current dataset for comparison
comboEpiAn = pd.concat([temp_dfEpiAn, SelEpiAn], ignore_index=True)




comboEpiAn['animalIndex'] = comboEpiAn['animalIndex'].astype(str) + "_" + comboEpiAn['dataset']

# Match each fish's bout and linear episode summaries by identity rather than
# relying on row order. Dataset remains in the key even though animalIndex is
# dataset-qualified above, making the intended cross-dataset separation explicit.
delta_key_columns = ['animalIndex', 'dataset', 'genotype', 'lineSet']
delta_episodes = ['01k01f', '02k20f']
delta_episode_rows = comboEpiAn.loc[
    comboEpiAn['episode'].isin(delta_episodes),
    delta_key_columns + ['episode', 'si'],
].copy()

duplicate_delta_rows = delta_episode_rows.duplicated(
    delta_key_columns + ['episode'],
    keep=False,
)
if duplicate_delta_rows.any():
    raise ValueError(
        'DeltaSI requires one summary row per fish and episode; found '
        f'{duplicate_delta_rows.sum()} duplicate rows.'
    )

delta_si_by_fish = (
    delta_episode_rows
    .pivot(index=delta_key_columns, columns='episode', values='si')
    .reset_index()
)
delta_si_by_fish.columns.name = None
for required_episode in delta_episodes:
    if required_episode not in delta_si_by_fish.columns:
        delta_si_by_fish[required_episode] = np.nan
delta_si_by_fish['DeltaSI'] = (
    delta_si_by_fish['02k20f'] - delta_si_by_fish['01k01f']
)

comboEpiAn = comboEpiAn.merge(
    delta_si_by_fish[delta_key_columns + ['DeltaSI']],
    on=delta_key_columns,
    how='left',
    validate='many_to_one',
)

comboEpiAn.genotype.unique()

In [ ]:
comboEpiAn

## Comparison of metrics *per experiment*

### Comparison of the different metrics across all Sel1 experiments

In [ ]:

# Comparison of the different metrics across all Sel1 experiments

# -----------------------------
# Data filtering settings
# -----------------------------
comboEpiAn['date'] = pd.to_datetime(
    comboEpiAn['date'],
    format='%d-%m-%Y',
    dayfirst=True,
    errors='coerce'
)

exclude_date1 = pd.Timestamp('2025-07-28')
exclude_date2 = pd.Timestamp('2025-12-31')


ix = (
    (comboEpiAn['episode'] == '02k20f') #only use shoaling attraction to bout-like stimulus
    & (comboEpiAn['genotype'].isin(['F2', 'hi', 'lo']))  # keep all genotypes but exclude na's or escapees from analysis
    & (comboEpiAn['boutDur'] < 4)  #filter extreme bout duration values
    #& ((comboEpiAn['date'] >= exclude_date1))  #only include experiments from 2026, when the setup was stable and optimized
    & ((comboEpiAn['date'] <= exclude_date1) | (comboEpiAn['date'] >= exclude_date2)) #only include experiments from early 2025 or experiments from 2026, when the setup was stable and optimized
    & comboEpiAn['lineSet'].str.startswith('Sel1', na=False) # keep only data from Sel1 for more direct comparison
    & ~comboEpiAn['lineSet'].str.startswith('Sel1_GC', na=False) # exclude GC lines
)



data_for_plot = (
    comboEpiAn[ix]
    .groupby(['animalIndex', 'genotype', 'lineSet'])
    .mean(numeric_only=True)
    .reset_index()
)

# Drop missing values
data_for_plot = data_for_plot.dropna(
    subset=['anSize', 'thigmoIndex', 'boutDur', 'si', 'avgSpeed_smooth']
)

data_for_plot.lineSet.unique()

missing_delta_si = int(data_for_plot['DeltaSI'].isna().sum())
if missing_delta_si:
    warnings.warn(
        f'Sel1 DeltaSI panel omitted {missing_delta_si} of {len(data_for_plot)} eligible fish '
        'because they lack a valid matched linear/bout SI pair.',
        UserWarning,
    )


# -----------------------------
# Plot settings
# -----------------------------
custom_palette = {'hi': "#FF8000", 'lo': "#0072B2", 'F2': "#009E28"}
order = ['hi', 'lo', 'F2']

variables = ['si', 'anSize', 'thigmoIndex', 'avgSpeed_smooth', 'boutDur', 'DeltaSI']
titles = ['SI', 'Animal Size', 'Thigmotaxis Index', 'Avg Speed (smooth)', 'Bout Duration', 'DeltaSI (bout - linear)']

# Optional thresholds
thresholds = {
    'avgSpeed_smooth': [1, 10],
    'thigmoIndex': [10, 30]
}

# -----------------------------
# NEW: Order lineSets by date
# -----------------------------
line_order = (
    comboEpiAn[ix]
    .groupby('lineSet')['date']
    .min()
    .sort_values()
    .index
)

# -----------------------------
# Create figure
# -----------------------------
fig, axes = plt.subplots(3, 2, figsize=(20, 15))
axes = axes.flatten()

for i, (var, title) in enumerate(zip(variables, titles)):
    ax = axes[i]

    # Mean Â± SD (pointplot)
    sns.pointplot(
        data=data_for_plot,
        x='lineSet',
        y=var,
        linestyle='none',
        errorbar='sd',
        hue='genotype',
        hue_order=order,
        palette=custom_palette,
        dodge=0.5,
        order=line_order,          # âœ… ordered by date
        legend=False,              # âœ… remove per-plot legends
        ax=ax
    )

    # Individual points (swarm)
    sns.swarmplot(
        data=data_for_plot,
        x='lineSet',
        y=var,
        size=5,
        alpha=0.65,
        hue='genotype',
        hue_order=order,
        palette=custom_palette,
        dodge=0.5,
        order=line_order,          # âœ… same ordering
        legend=False,              # âœ… remove per-plot legends
        ax=ax
    )

    # Capture legend handles ONCE
    if i == 0:
        handles, labels = ax.get_legend_handles_labels()

    # Threshold lines if defined
    if var in thresholds:
        for y in thresholds[var]:
            ax.axhline(y, color='red', linestyle='dotted', linewidth=1)
    if var == 'DeltaSI':
        ax.axhline(0, color='black', linestyle='dotted', linewidth=1)

    # Labels
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('Genotype')
    ax.set_ylabel(var)
    ax.tick_params(axis='x', rotation=90)

    if var == 'DeltaSI':
        ax.set_yticks(np.arange(-0.2, 0.55, 0.1))


# -----------------------------
# NEW: Global legend outside plots (fixed)
# -----------------------------
from matplotlib.lines import Line2D

# Create custom legend handles manually
legend_elements = [
    Line2D([0], [0], marker='o', color='w', label=gen,
           markerfacecolor=custom_palette[gen], markersize=8)
    for gen in order
]

fig.legend(
    handles=legend_elements,
    title='Genotype',
    loc='center left',
    bbox_to_anchor=(0.92, 0.5),   # âœ… closer to plots
    frameon=False                 # optional: cleaner look
)

# -----------------------------
# NEW: Adjust layout to make space for legend
# -----------------------------
plt.tight_layout(rect=[0, 0, 0.9, 1])  # âœ… less empty space than before

plt.show()


### Comparison of the different metrics across all Sel3 experiments

In [ ]:

# Comparison of the different metrics across all Sel3 experiments

# -----------------------------
# Data filtering settings
# -----------------------------
comboEpiAn['date'] = pd.to_datetime(
    comboEpiAn['date'],
    format='%d-%m-%Y',
    dayfirst=True,
    errors='coerce'
)

exclude_date1 = pd.Timestamp('2025-07-29')
exclude_date2 = pd.Timestamp('2025-12-31')
exclude_date3 = pd.Timestamp('2025-01-28')


ix = (
    (comboEpiAn['episode'] == '02k20f') #only use shoaling attraction to bout-like stimulus
    & (comboEpiAn['genotype'].isin(['F2', 'hi', 'lo']))  # keep all genotypes but exclude na's or escapees from analysis
    & (comboEpiAn['boutDur'] < 4)  #filter extreme bout duration values
    #& ((comboEpiAn['date'] >= exclude_date1))  #only include experiments from 2026, when the setup was stable and optimized
    & ((comboEpiAn['date'] <= exclude_date1) | (comboEpiAn['date'] >= exclude_date2)) #only include experiments from early 2025 or experiments from 2026, when the setup was stable and optimized
    & (comboEpiAn['lineSet'].str.startswith('Sel3', na=False) |
       comboEpiAn['lineSet'].str.startswith('TLSel3', na=False)
       ) # keep only data from Sel3 for more direct comparison
    & (comboEpiAn['date'] != exclude_date3)  #exclude data from first experiment in Jan 2025
    & ~comboEpiAn['lineSet'].str.startswith('Sel3_GC', na=False) # exclude GC lines
)



data_for_plot = (
    comboEpiAn[ix]
    .groupby(['animalIndex', 'genotype', 'lineSet'])
    .mean(numeric_only=True)
    .reset_index()
)

# Drop missing values
data_for_plot = data_for_plot.dropna(
    subset=['anSize', 'thigmoIndex', 'boutDur', 'si', 'avgSpeed_smooth']
)

data_for_plot.lineSet.unique()

missing_delta_si = int(data_for_plot['DeltaSI'].isna().sum())
if missing_delta_si:
    warnings.warn(
        f'Sel3 DeltaSI panel omitted {missing_delta_si} of {len(data_for_plot)} eligible fish '
        'because they lack a valid matched linear/bout SI pair.',
        UserWarning,
    )


# -----------------------------
# Plot settings
# -----------------------------
custom_palette = {'hi': "#FF8000", 'lo': "#0072B2", 'F2': "#009E28"}
order = ['hi', 'lo', 'F2']

variables = ['si', 'anSize', 'thigmoIndex', 'avgSpeed_smooth', 'boutDur', 'DeltaSI']
titles = ['SI', 'Animal Size', 'Thigmotaxis Index', 'Avg Speed (smooth)', 'Bout Duration', 'DeltaSI (bout - linear)']

# Optional thresholds
thresholds = {
    'avgSpeed_smooth': [1, 10],
    'thigmoIndex': [10, 30]
}

# -----------------------------
# NEW: Order lineSets by date
# -----------------------------
line_order = (
    comboEpiAn[ix]
    .groupby('lineSet')['date']
    .min()
    .sort_values()
    .index
)

# -----------------------------
# Create figure
# -----------------------------
fig, axes = plt.subplots(3, 2, figsize=(20, 15))
axes = axes.flatten()

for i, (var, title) in enumerate(zip(variables, titles)):
    ax = axes[i]

    # Mean Â± SD (pointplot)
    sns.pointplot(
        data=data_for_plot,
        x='lineSet',
        y=var,
        linestyle='none',
        errorbar='sd',
        hue='genotype',
        hue_order=order,
        palette=custom_palette,
        dodge=0.5,
        order=line_order,          # âœ… ordered by date
        legend=False,              # âœ… remove per-plot legends
        ax=ax
    )

    # Individual points (swarm)
    sns.swarmplot(
        data=data_for_plot,
        x='lineSet',
        y=var,
        size=5,
        alpha=0.65,
        hue='genotype',
        hue_order=order,
        palette=custom_palette,
        dodge=0.5,
        order=line_order,          # âœ… same ordering
        legend=False,              # âœ… remove per-plot legends
        ax=ax
    )

    # Capture legend handles ONCE
    if i == 0:
        handles, labels = ax.get_legend_handles_labels()

    # Threshold lines if defined
    if var in thresholds:
        for y in thresholds[var]:
            ax.axhline(y, color='red', linestyle='dotted', linewidth=1)
    if var == 'DeltaSI':
        ax.axhline(0, color='black', linestyle='dotted', linewidth=1)

    # Labels
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('Genotype')
    ax.set_ylabel(var)
    ax.tick_params(axis='x', rotation=90)

    if var == 'DeltaSI':
            ax.set_yticks(np.arange(-0.2, 0.55, 0.1))

# -----------------------------
# NEW: Global legend outside plots (fixed)
# -----------------------------
from matplotlib.lines import Line2D

# Create custom legend handles manually
legend_elements = [
    Line2D([0], [0], marker='o', color='w', label=gen,
           markerfacecolor=custom_palette[gen], markersize=8)
    for gen in order
]

fig.legend(
    handles=legend_elements,
    title='Genotype',
    loc='center left',
    bbox_to_anchor=(0.92, 0.5),   # âœ… closer to plots
    frameon=False                 # optional: cleaner look
)

# -----------------------------
# NEW: Adjust layout to make space for legend
# -----------------------------
plt.tight_layout(rect=[0, 0, 0.9, 1])  # âœ… less empty space than before

plt.show()